# Exploración de una sesión (EDA)

**Para qué sirve:** conocer la señal antes de confiar en el pipeline: calidad de la grabación, efecto de los
filtros, distribución de amplitudes y relación entre canales.

**Cómo usarlo:** cambia `SESSION_DIR` en la celda de parámetros y ejecuta todo (*Run All*). No escribe nada en
`data/`; solo lee la sesión y grafica.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
from pathlib import Path

# Las rutas de config/ y data/ son relativas a la raíz del repositorio.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pytest.ini").exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Raíz del repo:", ROOT)

In [ ]:
SESSION_DIR = "data/raw/subject_01/session_01"
SHOW_FULL_SESSION = True
T0_S = 0          # compatibilidad: el EDA completo no usa una ventana fija
DUR_S = None      # None = toda la sesión
CH_SEL = "EMG1"
UNIT_SCALE = 1e6


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy import signal

from src.config import load_yaml
from src.io import load_session
from src.segmentation import preprocess_emg
from src.analysis.signal_diagnostics import envelope_matrix

plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})

cfg = load_yaml("experiment/settings.yaml")
CH = cfg["channel_names"]
s = load_session(SESSION_DIR)
fs = s.report["emg"]["fs_est_hz"]
r = s.report

print(f"{s.subject} / {s.session}")
print(f"EMG: {r['emg']['n']:,} muestras | fs ≈ {fs:.1f} Hz | {r['emg']['duration_s']:.1f} s | "
      f"huecos: {r['emg']['gaps_gt_3x']} (máx. {r['emg']['max_gap_ms']:.1f} ms)")
print(f"Eventos de Unity: {r['events']['n']} | duración del protocolo: {r['events']['protocol_span_s']:.1f} s | "
      f"desfase de reloj: {r['events']['clock_skew_ms']:.3f} ms")
display(s.events["event"].value_counts().rename("n").to_frame().T)

In [ ]:
display(s.events[["seq", "event", "trial", "rep", "gesture", "reason"]].head(12))

## Señal cruda
Una ventana de la grabación sin procesar. Aquí se ve el **offset DC** (la señal no está centrada en 0) y
posible ruido de red de 60 Hz.

In [ ]:
t = (s.emg["t_ns"].to_numpy() - s.emg["t_ns"].iloc[0]) / 1e9
event_t = (s.events["t_ns"].to_numpy() - s.emg["t_ns"].iloc[0]) / 1e9
mask = np.ones_like(t, dtype=bool) if DUR_S is None else (t >= T0_S) & (t < T0_S + DUR_S)

fig, axes = plt.subplots(len(CH), 1, figsize=(18, 2.1 * len(CH)), sharex=True)
for ax, c in zip(axes, CH):
    ax.plot(t[mask], s.emg[c].to_numpy()[mask] * UNIT_SCALE, lw=0.5, color="tab:blue")
    for et in event_t:
        ax.axvline(et, color="tab:red", lw=0.35, alpha=0.35)
    ax.set_ylabel(f"{c} (µV)")
axes[-1].set_xlabel("s desde el inicio del EMG")
fig.suptitle("EMG completo + eventos de Unity" if DUR_S is None else f"EMG ({T0_S:.0f}–{T0_S + DUR_S:.0f} s) + eventos de Unity")
fig.tight_layout()


## Efecto del filtrado
Remoción de DC, notch de 60 Hz y pasa-banda 20–450 Hz (parámetros en `config/experiment/settings.yaml`).
En el espectro, el pico de 60 Hz del crudo debe desaparecer y la energía por debajo de 20 Hz debe caer.

In [ ]:
filt = preprocess_emg(s.emg, fs, cfg["preprocessing"], CH)

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(t[m], s.emg[CH_SEL].to_numpy()[m] * UNIT_SCALE, lw=0.5, color="gray", label="crudo")
ax.plot(t[m], filt[CH_SEL].to_numpy()[m] * UNIT_SCALE, lw=0.6, color="crimson", label="filtrado")
ax.set_ylabel(f"{CH_SEL} (µV)"); ax.set_xlabel("s desde el inicio del EMG"); ax.legend()
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, len(CH), figsize=(16, 3.4), sharey=True)
for ax, c in zip(axes, CH):
    f, p_raw = signal.welch(s.emg[c].to_numpy(), fs, nperseg=2048)
    _, p_flt = signal.welch(filt[c].to_numpy(), fs, nperseg=2048)
    ax.semilogy(f, p_raw, alpha=0.7, label="crudo")
    ax.semilogy(f, p_flt, label="filtrado")
    ax.axvline(60, color="k", ls=":", lw=1)
    ax.set_xlim(0, 500); ax.set_title(c); ax.set_xlabel("Hz")
axes[0].set_ylabel("PSD (V²/Hz)"); axes[0].legend()
fig.tight_layout()

## Amplitud y relación entre canales
- **Curtosis alta** = colas pesadas: la señal tiene ráfagas de actividad sobre un fondo de ruido (esperable en EMG).
- **Correlación de envolventes cercana a 1 entre todos los canales** sugiere artefactos comunes (movimiento del
  cable, referencia) o diafonía, no actividad muscular independiente.

In [ ]:
stats = (filt[CH].describe(percentiles=[0.01, 0.5, 0.99]).T[["min", "1%", "50%", "99%", "max", "std"]]
         * UNIT_SCALE).round(2)
stats["curtosis (exceso)"] = filt[CH].kurt().round(1)
display(stats)

fig, axes = plt.subplots(1, len(CH), figsize=(16, 2.8), sharey=True)
for ax, c in zip(axes, CH):
    ax.hist(filt[c] * UNIT_SCALE, bins=120, log=True)
    ax.set_title(c); ax.set_xlabel("µV")
axes[0].set_ylabel("muestras")
fig.tight_layout()

In [ ]:
env = envelope_matrix(filt[CH].to_numpy(), fs)
corr = pd.DataFrame(env, columns=CH).corr()

fig, ax = plt.subplots(figsize=(4.6, 4))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(CH)), CH); ax.set_yticks(range(len(CH)), CH); ax.grid(False)
for i in range(len(CH)):
    for j in range(len(CH)):
        ax.text(j, i, f"{corr.iat[i, j]:.2f}", ha="center", va="center")
fig.colorbar(im, ax=ax); ax.set_title("Correlación de envolventes")
fig.tight_layout()

## IMU (opcional)
Si la sesión incluye Baiobit, el giroscopio ayuda a detectar movimientos de cabeza que pueden contaminar el EMG.

In [ ]:
if s.imu is None:
    print("Esta sesión no incluye datos del Baiobit.")
else:
    ti = (s.imu["t_ns"].to_numpy() - s.emg["t_ns"].iloc[0]) / 1e9
    mi = np.ones_like(ti, dtype=bool) if DUR_S is None else (ti >= T0_S) & (ti < T0_S + DUR_S)
    if not np.any(mi):
        print(f"No hay muestras del IMU en el rango [{T0_S}, {T0_S + (DUR_S if DUR_S is not None else 0)}] s.")
    else:
        fig, ax = plt.subplots(figsize=(14, 3))
        for c in ("gx", "gy", "gz"):
            ax.plot(ti[mi], s.imu[c].to_numpy()[mi], lw=0.7, label=c)
        ax.set_xlabel("s desde el inicio del EMG")
        ax.set_ylabel("giroscopio")
        ax.legend()
        fig.tight_layout()

---
**Siguiente paso:** `notebooks/label_validation/01_validar_etiquetas.ipynb` para comprobar que las etiquetas de
Unity coinciden con la actividad muscular.